# EDA 01 — Raw metadata files exploration

This notebook performs an initial exploratory analysis of the raw metadata files used in the project.

The objective is to inspect the structure, dimensions, columns, missing values and basic content of the three original CSV files stored in `data/raw/metadata`.

No preprocessing or feature engineering is performed in this notebook. The goal is only to understand the raw input data before defining the preprocessing pipeline.

In [1]:
# Libraries
from skin_lesion_ai.utils.config import load_raw_metadata
import pandas as pd

In [2]:
# Loading data
df1, df2, df3 = load_raw_metadata()
print("df1 - ground_truth:", df1.shape)
print("df2 - supplement:", df2.shape)
print("df3 - metadata:", df3.shape)

/Users/carlesraichbros/my-image-classifier/src/skin_lesion_ai/utils/config.py:43: DtypeWarning: Columns (0: iddx_5) have mixed types. Specify dtype option on import or set low_memory=False.
  supplement = pd.read_csv(path(raw["supplement_csv"]))


df1 - ground_truth: (401059, 2)
df2 - supplement: (401059, 13)
df3 - metadata: (401059, 42)


## ground_truth (df1)

In [3]:
# Data Wrangler
df1

,isic_id,malignant
0,ISIC_0015670,0.0
1,ISIC_0015845,0.0
2,ISIC_0015864,0.0
3,ISIC_0015902,0.0
4,ISIC_0024200,0.0
...,...,...
401054,ISIC_9999937,0.0
401055,ISIC_9999951,0.0
401056,ISIC_9999960,0.0
401057,ISIC_9999964,0.0


In [4]:
# malignant col
df1["malignant"].unique().tolist()

[0.0, 1.0]

In [5]:
# malignant proportions
malignant_summary = (
    df1["malignant"]
    .value_counts()
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / x["count"].sum(), 2))
)

malignant_summary.loc["Total"] = [
    malignant_summary["count"].sum(),
    malignant_summary["percentage"].sum(),
]

malignant_summary

,count,percentage
malignant,,
0.0,400666.0,99.9
1.0,393.0,0.1
Total,401059.0,100.0


### ground_truth (df1) summary

Basic lesion labeling. 

- **Rows**: 401059. No duplicated rows.
- **Cols**: 2
- **Col names**: [isic_id, malignant]

    - *isic_id*: unique id per lesion (pkey). 0 NA.  

    - *malignant*: malignancy flag per lesion. 0 NA. Two values: 
        - 0 = not malignant (400,666 counts; 99.9%). 
        - 1 = malignant (393 counts; 0.1%).

## supplement (df2)

In [6]:
# Data Wrangler
df2

,isic_id,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,Memorial Sloan Kettering Cancer Center,CC-BY,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455
2,ISIC_0015864,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.804040
3,ISIC_0015902,ACEMID MIA,CC-0,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.989998
4,ISIC_0024200,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,70.442510
...,...,...,...,...,...,...,...,...,...,...,...,...,...
401054,ISIC_9999937,"Department of Dermatology, Hospital Clínic de ...",CC-BY-NC,IL_9520694,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999988
401055,ISIC_9999951,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999820
401056,ISIC_9999960,"Frazer Institute, The University of Queensland...",CC-BY,IL_9852274,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999416
401057,ISIC_9999964,University Hospital of Basel,CC-BY-NC,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,100.000000


In [7]:
df2.columns.tolist()

['isic_id',
 'attribution',
 'copyright_license',
 'lesion_id',
 'iddx_full',
 'iddx_1',
 'iddx_2',
 'iddx_3',
 'iddx_4',
 'iddx_5',
 'mel_mitotic_index',
 'mel_thick_mm',
 'tbp_lv_dnn_lesion_confidence']

In [8]:
# isic_id col
isic_id_df1 = set(df1["isic_id"])
isic_id_df2 = set(df2["isic_id"])

print("Only in df1:", isic_id_df1 - isic_id_df2)
print("Only in df2:", isic_id_df2 - isic_id_df1)

Only in df1: set()
Only in df2: set()


In [9]:
# attribution col
df2["attribution"].unique().tolist()

['Memorial Sloan Kettering Cancer Center',
 'ACEMID MIA',
 'Department of Dermatology, Hospital Clínic de Barcelona',
 'University Hospital of Basel',
 'Frazer Institute, The University of Queensland, Dermatology Research Centre',
 'Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris',
 'ViDIR Group, Department of Dermatology, Medical University of Vienna']

In [10]:
# attribution proportions
attribution_summary = (
    df2["attribution"]
    .value_counts()
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / x["count"].sum(), 2))
)

attribution_summary.loc["Total"] = [
    attribution_summary["count"].sum(),
    attribution_summary["percentage"].sum(),
]

attribution_summary

,count,percentage
attribution,,
Memorial Sloan Kettering Cancer Center,129068.0,32.18
"Department of Dermatology, Hospital Clínic de Barcelona",105724.0,26.36
University Hospital of Basel,65218.0,16.26
"Frazer Institute, The University of Queensland, Dermatology Research Centre",51768.0,12.91
ACEMID MIA,28665.0,7.15
"ViDIR Group, Department of Dermatology, Medical University of Vienna",12640.0,3.15
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",7976.0,1.99
Total,401059.0,100.00


In [11]:
# copyright_licese col
df2["copyright_license"].unique().tolist()

['CC-BY', 'CC-0', 'CC-BY-NC']

In [12]:
# license per site
pd.crosstab(df2["attribution"], df2["copyright_license"])

copyright_license,CC-0,CC-BY,CC-BY-NC
attribution,,,
ACEMID MIA,28665,0,0
"Department of Dermatology, Hospital Clínic de Barcelona",0,0,105724
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",0,7976,0
"Frazer Institute, The University of Queensland, Dermatology Research Centre",0,51768,0
Memorial Sloan Kettering Cancer Center,0,129068,0
University Hospital of Basel,0,0,65218
"ViDIR Group, Department of Dermatology, Medical University of Vienna",0,0,12640


In [13]:
# lesion_id col

n_null = df2["lesion_id"].isna().sum()
n_non_null = df2["lesion_id"].notna().sum()
n_dup = df2.loc[df2["lesion_id"].notna(), "lesion_id"].duplicated().sum()

lesion_id_summary = pd.DataFrame(
    {
        "count": [n_null, n_non_null, len(df2), n_dup],
        "percentage": [
            round(100 * n_null / len(df2), 2),
            round(100 * n_non_null / len(df2), 2),
            100.0,
            round(100 * n_dup / n_non_null, 2),
        ],
    },
    index=["Null", "Non-null", "Total", "Non-null duplicated"],
)

lesion_id_summary

,count,percentage
Null,379001,94.5
Non-null,22058,5.5
Total,401059,100.0
Non-null duplicated,0,0.0


In [14]:
# lesion_id per site
(
    df2.assign(has_lesion_id=df2["lesion_id"].notna())
    .groupby("attribution")["has_lesion_id"]
    .agg(total="size", with_lesion_id="sum")
    .assign(percentage=lambda x: round(100 * x["with_lesion_id"] / x["total"], 2))
    .sort_values("percentage", ascending=False)
)

,total,with_lesion_id,percentage
attribution,,,
"Frazer Institute, The University of Queensland, Dermatology Research Centre",51768,9056,17.49
ACEMID MIA,28665,1792,6.25
Memorial Sloan Kettering Cancer Center,129068,6212,4.81
University Hospital of Basel,65218,3004,4.61
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",7976,167,2.09
"Department of Dermatology, Hospital Clínic de Barcelona",105724,1720,1.63
"ViDIR Group, Department of Dermatology, Medical University of Vienna",12640,107,0.85


In [15]:
# iddx_* cols

# iddx_1 col
iddx_1_summary = (
    df2["iddx_1"]
    .value_counts(dropna=False)
    .rename_axis("iddx_1")
    .reset_index(name="count")
)

iddx_1_summary["percentage"] = round(100 * iddx_1_summary["count"] / len(df2), 2)

iddx_1_summary

,iddx_1,count,percentage
0,Benign,400552,99.87
1,Malignant,393,0.10
2,Indeterminate,114,0.03


In [16]:
# iddx_2 col
iddx_2_summary = (
    df2["iddx_2"]
    .value_counts(dropna=False)
    .rename_axis("iddx_2")
    .reset_index(name="count")
)

iddx_2_summary["percentage"] = round(100 * iddx_2_summary["count"] / len(df2), 2)

iddx_2_summary

,iddx_2,count,percentage
0,NaN,399991,99.73
1,Benign melanocytic proliferations,443,0.11
2,Malignant adnexal epithelial proliferations - ...,163,0.04
3,Malignant melanocytic proliferations (Melanoma),157,0.04
4,Benign epidermal proliferations,83,0.02
5,Indeterminate melanocytic proliferations,75,0.02
6,Malignant epidermal proliferations,73,0.02
7,Indeterminate epidermal proliferations,39,0.01
8,Benign soft tissue proliferations - Fibro-hist...,15,0.00
9,Inflammatory or infectious diseases,7,0.00


In [17]:
for lvl1 in sorted(df2["iddx_1"].dropna().unique()):
    print(f"\n{lvl1}")

    children = df2.loc[df2["iddx_1"] == lvl1, "iddx_2"].dropna().value_counts()

    for child, n in children.items():
        print(f"  └─ {child}: {n}")


Benign
  └─ Benign melanocytic proliferations: 443
  └─ Benign epidermal proliferations: 83
  └─ Benign soft tissue proliferations - Fibro-histiocytic: 15
  └─ Inflammatory or infectious diseases: 7
  └─ Flat melanotic pigmentations - not melanocytic nevus: 5
  └─ Benign soft tissue proliferations - Vascular: 3
  └─ Cysts: 2
  └─ Benign adnexal epithelial proliferations - Follicular: 2
  └─ Benign adnexal epithelial proliferations - Apocrine or Eccrine: 1

Indeterminate
  └─ Indeterminate melanocytic proliferations: 75
  └─ Indeterminate epidermal proliferations: 39

Malignant
  └─ Malignant adnexal epithelial proliferations - Follicular: 163
  └─ Malignant melanocytic proliferations (Melanoma): 157
  └─ Malignant epidermal proliferations: 73


In [18]:
(
    df2.loc[df2["iddx_2"].isna(), "iddx_1"]
    .value_counts(dropna=False)
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / len(df2), 2))
)

,count,percentage
iddx_1,,
Benign,399991,99.73


In [19]:
(df2.loc[df2["iddx_2"].isna(), "iddx_1"] == "Benign").all()

np.True_

In [20]:
# iddx_3 col
iddx_3_summary = (
    df2["iddx_3"]
    .value_counts(dropna=False)
    .rename_axis("iddx_3")
    .reset_index(name="count")
)

iddx_3_summary["percentage"] = round(100 * iddx_3_summary["count"] / len(df2), 2)

iddx_3_summary

,iddx_3,count,percentage
0,NaN,399994,99.73
1,Nevus,443,0.11
2,Basal cell carcinoma,163,0.04
3,Melanoma in situ,80,0.02
4,Atypical melanocytic neoplasm,64,0.02
5,Melanoma Invasive,63,0.02
6,Seborrheic keratosis,57,0.01
7,Squamous cell carcinoma in situ,49,0.01
8,Solar or actinic keratosis,39,0.01
9,"Squamous cell carcinoma, Invasive",22,0.01


In [21]:
# iddx_4 col
iddx_4_summary = (
    df2["iddx_4"]
    .value_counts(dropna=False)
    .rename_axis("iddx_4")
    .reset_index(name="count")
)

iddx_4_summary["percentage"] = round(100 * iddx_4_summary["count"] / len(df2), 2)

iddx_4_summary

,iddx_4,count,percentage
0,NaN,400508,99.86
1,"Nevus, Atypical, Dysplastic, or Clark",228,0.06
2,"Basal cell carcinoma, Nodular",98,0.02
3,"Basal cell carcinoma, Superficial",48,0.01
4,"Melanoma Invasive, Superficial spreading",37,0.01
5,"Nevus, NOS, Compound",30,0.01
6,"Nevus, NOS, Dermal",20,0.00
7,"Melanoma in situ, Lentigo maligna type",12,0.00
8,"Melanoma in situ, associated with a nevus",12,0.00
9,"Nevus, NOS, Junctional",10,0.00


In [22]:
# iddx_5 col
iddx_5_summary = (
    df2["iddx_5"]
    .value_counts(dropna=False)
    .rename_axis("iddx_5")
    .reset_index(name="count")
)

iddx_5_summary["percentage"] = round(100 * iddx_5_summary["count"] / len(df2), 2)

iddx_5_summary

,iddx_5,count,percentage
0,NaN,401058,100.0
1,"Blue nevus, Cellular",1,0.0


In [23]:
iddx_cols = ["iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5"]

iddx_hierarchy_summary = (
    df2.groupby(iddx_cols, dropna=False).size().reset_index(name="count")
)

iddx_hierarchy_summary["percentage"] = round(
    100 * iddx_hierarchy_summary["count"] / len(df2), 4
)

iddx_hierarchy_summary = iddx_hierarchy_summary.sort_values(
    ["iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5"], na_position="first"
)

iddx_hierarchy_summary

,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,count,percentage
27,Benign,NaN,NaN,NaN,NaN,399991,99.7337
0,Benign,Benign adnexal epithelial proliferations - Apo...,Hidradenoma,NaN,NaN,1,0.0002
1,Benign,Benign adnexal epithelial proliferations - Fol...,NaN,NaN,NaN,2,0.0005
2,Benign,Benign epidermal proliferations,Lichen planus like keratosis,NaN,NaN,11,0.0027
3,Benign,Benign epidermal proliferations,Pigmented benign keratosis,NaN,NaN,3,0.0007
5,Benign,Benign epidermal proliferations,Seborrheic keratosis,NaN,NaN,56,0.0140
4,Benign,Benign epidermal proliferations,Seborrheic keratosis,"Seborrheic keratosis, Clonal",NaN,1,0.0002
6,Benign,Benign epidermal proliferations,Solar lentigo,NaN,NaN,12,0.0030
17,Benign,Benign melanocytic proliferations,Nevus,NaN,NaN,141,0.0352
7,Benign,Benign melanocytic proliferations,Nevus,Blue nevus,"Blue nevus, Cellular",1,0.0002


In [24]:
# iddx_full col

iddx_cols = ["iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5"]
df2["iddx_reconstructed"] = (
    df2[iddx_cols]
    .fillna("")
    .apply(lambda x: "::".join([v for v in x if v != ""]), axis=1)
)

(df2["iddx_full"] == df2["iddx_reconstructed"]).value_counts()

True    401059
Name: count, dtype: int64

In [ ]:
# manual tag and iddx_2 col check 1
manual_iddx_2_summary = (
    df2.loc[df2["lesion_id"].notna(), "iddx_2"]
    .value_counts(dropna=False)
    .rename_axis("iddx_2")
    .reset_index(name="count")
)

manual_iddx_2_summary["percentage_manual_tag"] = round(
    100 * manual_iddx_2_summary["count"] / df2["lesion_id"].notna().sum(), 2
)

manual_iddx_2_summary

,iddx_2,count,percentage_manual_tag
0,NaN,20990,95.16
1,Benign melanocytic proliferations,443,2.01
2,Malignant adnexal epithelial proliferations - ...,163,0.74
3,Malignant melanocytic proliferations (Melanoma),157,0.71
4,Benign epidermal proliferations,83,0.38
5,Indeterminate melanocytic proliferations,75,0.34
6,Malignant epidermal proliferations,73,0.33
7,Indeterminate epidermal proliferations,39,0.18
8,Benign soft tissue proliferations - Fibro-hist...,15,0.07
9,Inflammatory or infectious diseases,7,0.03


In [26]:
#  manual tags vs iddx_2 null / non-null

has_manual_tag = df2["lesion_id"].notna()
iddx_2_status = (
    df2["iddx_2"]
    .notna()
    .map(
        {
            True: "iddx_2 non-null",
            False: "iddx_2 null",
        }
    )
)

manual_by_iddx_2_status = (
    df2.assign(
        has_manual_tag=has_manual_tag,
        iddx_2_status=iddx_2_status,
    )
    .groupby("iddx_2_status")
    .agg(
        total_cases=("isic_id", "count"),
        manual_tag_cases=("has_manual_tag", "sum"),
    )
    .assign(
        percentage_manual_tag=lambda x: round(
            100 * x["manual_tag_cases"] / x["total_cases"], 2
        )
    )
)

manual_by_iddx_2_status

,total_cases,manual_tag_cases,percentage_manual_tag
iddx_2_status,,,
iddx_2 non-null,1068,1068,100.00
iddx_2 null,399991,20990,5.25


In [28]:
# manual tags vs iddx_1 in non null iddx_2
manual_by_iddx_1_within_non_null_iddx_2 = (
    df2.loc[df2["iddx_2"].notna()]
    .assign(has_manual_tag=lambda x: x["lesion_id"].notna())
    .groupby("iddx_1")
    .agg(
        total_cases=("isic_id", "count"),
        manual_tag_cases=("has_manual_tag", "sum"),
    )
    .assign(
        percentage_manual_tag=lambda x: round(
            100 * x["manual_tag_cases"] / x["total_cases"], 2
        ),
    )
    .sort_values("manual_tag_cases", ascending=False)
)

manual_by_iddx_1_within_non_null_iddx_2

,total_cases,manual_tag_cases,percentage_manual_tag
iddx_1,,,
Benign,561,561,100.0
Malignant,393,393,100.0
Indeterminate,114,114,100.0


In [31]:
# mel_thick_mm and mel_mitotic_index cols
for col in ["mel_thick_mm", "mel_mitotic_index"]:
    print(f"\n=== {col} ===")

    display(
        df2.loc[df2[col].notna(), "iddx_full"]
        .value_counts(dropna=False)
        .to_frame("count")
    )


=== mel_thick_mm ===


,count
iddx_full,
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Superficial spreading",37
Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive,13
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Associated with a nevus",7
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, On chronically sun-exposed skin or lentigo maligna melanoma",5
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Nodular",1



=== mel_mitotic_index ===


,count
iddx_full,
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Superficial spreading",35
Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive,7
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Associated with a nevus",5
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, On chronically sun-exposed skin or lentigo maligna melanoma",5
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Nodular",1


In [35]:
for col in ["mel_thick_mm", "mel_mitotic_index"]:
    print(f"\n=== {col} ===")

    summary = pd.DataFrame(
        {
            "count": [df2[col].count()],
            "missing": [df2[col].isna().sum()],
            "missing_%": [round(100 * df2[col].isna().sum() / len(df2), 4)],
        }
    )

    display(summary)

    display(df2[col].describe())


=== mel_thick_mm ===


,count,missing,missing_%
0,63,400996,99.9843


count    63.000000
mean      0.670952
std       0.792798
min       0.200000
25%       0.300000
50%       0.400000
75%       0.600000
max       5.000000
Name: mel_thick_mm, dtype: float64


=== mel_mitotic_index ===


,count,missing,missing_%
0,53,401006,99.9868


count         53
unique         7
top       0/mm^2
freq          22
Name: mel_mitotic_index, dtype: object

In [ ]:
# tbp_lv_dnn_lesion_confidence col (lesion confidence)

df2["tbp_lv_dnn_lesion_confidence"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

count    4.010590e+05
mean     9.716220e+01
std      8.995782e+00
min      1.261082e-16
1%       5.542216e+01
5%       7.852168e+01
25%      9.966882e+01
50%      9.999459e+01
75%      9.999996e+01
95%      1.000000e+02
99%      1.000000e+02
max      1.000000e+02
Name: tbp_lv_dnn_lesion_confidence, dtype: float64

In [ ]:
# lesion confidence vs iddx_1
(df2.groupby("iddx_1")["tbp_lv_dnn_lesion_confidence"].describe())

,count,mean,std,min,25%,50%,75%,max
iddx_1,,,,,,,,
Benign,400552.0,97.180388,8.910078,1.261082e-16,99.670264,99.994610,99.999960,100.0
Indeterminate,114.0,87.500639,29.704617,6.865393e-08,99.057932,99.988927,99.999980,100.0
Malignant,393.0,81.431493,33.805649,2.256579e-06,83.221790,99.684890,99.995804,100.0


In [40]:
# lesion confidence vs iddx_2
(
    df2.loc[df2["iddx_2"].notna()]
    .groupby("iddx_2")["tbp_lv_dnn_lesion_confidence"]
    .describe()
    .sort_values("mean", ascending=False)
)

,count,mean,std,min,25%,50%,75%,max
iddx_2,,,,,,,,
Benign adnexal epithelial proliferations - Apocrine or Eccrine,1.0,99.999452,NaN,9.999945e+01,99.999452,99.999452,99.999452,99.999452
Benign adnexal epithelial proliferations - Follicular,2.0,99.708015,0.412930,9.941603e+01,99.562022,99.708015,99.854007,100.000000
Indeterminate melanocytic proliferations,75.0,96.895410,14.955200,1.560452e+01,99.975859,99.999809,100.000000,100.000000
Benign melanocytic proliferations,443.0,95.638540,17.780352,5.548416e-09,99.986955,99.999870,100.000000,100.000000
Flat melanotic pigmentations - not melanocytic nevus,5.0,92.276946,11.506631,7.383386e+01,88.043220,99.546885,99.961310,99.999450
Malignant melanocytic proliferations (Melanoma),157.0,87.114046,30.717087,2.255627e-05,99.112090,99.976740,99.999690,100.000000
Malignant adnexal epithelial proliferations - Follicular,163.0,82.903924,31.831764,3.286650e-06,88.965660,99.348181,99.988873,100.000000
Benign epidermal proliferations,83.0,82.760935,32.699677,6.066522e-05,81.150359,99.876201,99.995571,100.000000
Cysts,2.0,82.038397,24.862577,6.445790e+01,73.248148,82.038397,90.828645,99.618894


### supplement (df2) summary

Detailed lesion labeling.

- **Rows**: 401059. No duplicated rows.
- **Cols**: 13.
- **Col names**: ['isic_id','attribution','copyright_license','lesion_id','iddx_full','iddx_1''iddx_2','iddx_3','iddx_4','iddx_5','mel_mitotic_index','mel_thick_mm', 'tbp_lv_dnn_lesion_confidence'].

   - *isic_id*: unique id per lesion (pkey). 0 NA. 100% match with df1.

   - *attribution*: lesion site of origin. 0 NA. 7 sites (DESC order):

      - Memorial Sloan Kettering Cancer Center (USA): 129,086 lesions (32.18%).
      - Department of Dermatology, Hospital Clínic de Barcelona (Spain): 105,274 lesions (26.36%).
      - University Hospital of Basel (Switzerland): 65,218 lesions (16.26%).
      - Frazer Institute, The University of Queensland, Dermatology Research Centre (Australia): 65,218 lesions (12.91%).
      - ACEMID MIA (Australian Centre of Excellence in Melanoma Imaging and Diagnosis - Melanoma Institute Australia), which includes Alfred Hospital and FNQH Cairn (Australia): 28,665 lesions (7.15%).
      - ViDIR Group, Department of Dermatology, Medical University of Vienna (Austria): 12,640 lesions (3.15%).
      - Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris (Greece): 7,976 lesions (1.99%).

   - *copyright_license*: 3 types of copyright license. 0 NA.

      - *CC-0 (Creative Commons Zero)*: Public domain dedication. The data can be used, modified, and redistributed without attribution and without restrictions. This applies to:
         - ACEMID MIA

      - *CC-BY (Creative Commons Attribution)*: The data can be used, modified, and redistributed, including for commercial purposes, provided that appropriate attribution is given to the original source. This applies to:
         - Memorial Sloan Kettering Cancer Center
         - Frazer Institute, The University of Queensland, Dermatology Research Centre
         - Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris

      - *CC-BY-NC (Creative Commons Attribution–NonCommercial)*: The data can be used, modified, and redistributed with attribution, but commercial use is not permitted. This applies to:
         - Department of Dermatology, Hospital Clínic de Barcelona
         - University Hospital of Basel
         - ViDIR Group, Department of Dermatology, Medical University of Vienna

   - *lesion_id*: id for some lesions indicating it was a manual tag. Non null and unique for 20,058 cases (5.5%), the rest of lesions have a null lesion_id. All sites present manual tags.

   - *iddx_full / iddx_1-5*: hierarchical lesion diagnosis labels.

      The diagnosis variables follow a hierarchical taxonomy of increasing specificity. `iddx_1` contains the broadest diagnostic category, while `iddx_5` contains the most specific diagnosis when available. `iddx_full` stores the complete hierarchical path as a single string.

      Three first-level (`iddx_1`) diagnostic groups are present:

      - *Benign*: 400,552 lesions (99.87%).
      - *Malignant*: 393 lesions (0.10%).
      - *Indeterminate*: 114 lesions (0.03%).

      The second-level diagnosis (`iddx_2`) contains the following categories:

      - *Benign iddx_1*:
         - NULL (no biopsy): 399,991 lesions (99.73%).
         - Benign melanocytic proliferations: 443 lesions (0.11%).
         - Benign epidermal proliferations: 83 lesions (0.02%).
         - Benign soft tissue proliferations - Fibro-histiocytic: 15 lesions (<0.01%).
         - Inflammatory or infectious diseases: 7 lesions (<0.01%).
         - Flat melanotic pigmentations - not melanocytic nevus: 5 lesions (<0.01%).
         - Benign soft tissue proliferations - Vascular: 3 lesions (<0.01%).
         - Cysts: 2 lesions (<0.01%).
         - Benign adnexal epithelial proliferations - Follicular: 2 lesions (<0.01%).
         - Benign adnexal epithelial proliferations - Apocrine or Eccrine: 1 lesion (<0.01%).

      - *Indeterminate iddx_1*:
         - Indeterminate melanocytic proliferations: 75 lesions (0.02%).
         - Indeterminate epidermal proliferations: 39 lesions (0.01%).

      - *Malignant iddx_1*:
         - Malignant adnexal epithelial proliferations - Follicular: 163 lesions (0.04%).
         - Malignant melanocytic proliferations (Melanoma): 157 lesions (0.04%).
         - Malignant epidermal proliferations: 73 lesions (0.02%).

      The deeper diagnostic levels (`iddx_3-5`) further refine the diagnosis hierarchy for some non null `iddx_2` lesions into increasingly specific entities and histopathological subtypes (see `iddx_hierarchy_summary`). Across the dataset, 52 unique diagnostic paths are observed. 
      
      Only 1,068 lesions (0.27% of the dataset) contain a biopsy-confirmed detailed diagnosis beyond the first diagnostic level (`iddx_2-5`). Among these biopsied lesions, the five most frequent second-level diagnostic categories are:

      - Benign melanocytic proliferations: 443 lesions (41.48%).
      - Malignant adnexal epithelial proliferations - Follicular: 163 lesions (15.26%).
      - Malignant melanocytic proliferations (Melanoma): 157 lesions (14.70%).
      - Benign epidermal proliferations: 83 lesions (7.77%).
      - Indeterminate melanocytic proliferations: 75 lesions (7.02%).

      Together, these categories account for 86.24% of all biopsied lesions. 
      
      An important observation is that all lesions with detailed diagnostic information (`iddx_2-5`) are associated with a non-null `lesion_id`. In other words, 100% of lesions with a detailed diagnosis belong to the manually tagged subset of the dataset. Conversely, among lesions lacking a detailed diagnosis (`iddx_2 = NA`), only 20,990 cases (5.25%) contain a manual lesion identifier. This suggests a strong association between manual annotation and the availability of detailed diagnostic information.

   - *mel_thick_mm*: thickness in depth of melanoma invasion (Breslow thickness, in millimetres) derived from histopathological examination (lesions that underwent biopsy), measurement to assess melanoma severity. 

      The variable is available for only 63 melanoma lesions (approximately 40.1% of all melanoma cases). The remaining 400,996 observations (99.984% of the dataset) are missing.

      All non-null observations correspond to invasive melanoma diagnoses. Values range from 0.2 mm to 5.0 mm, with a median thickness of 0.4 mm (IQR: 0.3–0.6 mm). The distribution is right-skewed, with a small number of thicker lesions reaching up to 5 mm.

   - *mel_mitotic_index*: mitotic index of invasive malignant melanomas derived from histopathological examination (lesions that underwent biopsy), measurement to assess melanoma severity. 

      The variable is available for only 53 melanoma lesions (approximately 33.8% of all melanoma cases). The remaining 401,006 observations (99.987% of the dataset) are missing.

      All non-null observations correspond to invasive melanoma diagnoses. The variable is stored as a categorical measurement (e.g. "0/mm²") and contains seven distinct values. The most frequent category is 0/mm², representing 22 of the 53 recorded cases (41.5%).

      Together, `mel_thick_mm` and `mel_mitotic_index` provide histopathological information that is only available for a very small subset of invasive melanomas within the dataset.

   - *tbp_lv_dnn_lesion_confidence*: lesion confidence score on a 0–100 scale. Unspecified / unknown (probably internal) score. 0 NA.

      Values range from 0 to 100, with a mean of 97.16 and a median of 99.99. The distribution is heavily concentrated near the upper bound of the scale, with 75% of lesions having scores above 99.999 and 95% above 100.

      This indicates that the vast majority of lesions were assigned a very high confidence score. Based on its distribution and the dataset documentation, the variable appears to represent confidence in lesion detection rather than a direct measure of malignancy risk.


## metadata (df3)